In [1]:
import json
from pathlib import Path

# Paths
BASE_DIR = Path("../data")
EXTRACTED_QUESTIONS = BASE_DIR / "extracted_questions.json"
PARSED_ANSWERS = BASE_DIR / "parsed_quiz_test_data/parsed_quiz_test_data.json"
SIGNS_DIR = BASE_DIR / "individual_signs"
PRIORITIES_DIR = BASE_DIR / "individual_priorities"
OUTPUT_FILE = BASE_DIR / "final_quiz_data.json"

print("✓ Setup complete")

✓ Setup complete


## Load Source Data

In [2]:
# Load questions (French & Arabic)
with open(EXTRACTED_QUESTIONS, 'r', encoding='utf-8') as f:
    questions_data = json.load(f)

# Load answers from parsed data
with open(PARSED_ANSWERS, 'r', encoding='utf-8') as f:
    parsed_data = json.load(f)

# Extract answer arrays
signs_answers_all = parsed_data.get("الإشارات", [])  # 20 arrays of 16 answers each
priorities_answers_all = parsed_data.get("أولوية المرور", [])  # 20 arrays of 8 answers each
questions_answers_all = parsed_data.get("الأسئلة", [])  # 20 arrays of 6 answers each

print(f"✓ Loaded {len(questions_data)} tests with questions")
print(f"✓ Loaded {len(signs_answers_all)} tests of road signs answers")
print(f"✓ Loaded {len(priorities_answers_all)} tests of priorities answers")
print(f"✓ Loaded {len(questions_answers_all)} tests of questions answers")

✓ Loaded 20 tests with questions
✓ Loaded 20 tests of road signs answers
✓ Loaded 20 tests of priorities answers
✓ Loaded 20 tests of questions answers


## Build Final Structure

In [3]:
# Final data structure
final_data = {}

for test_num in range(1, 21):
    test_name = f"test-{test_num:02d}"
    test_index = test_num - 1  # Array index (0-based)
    
    print(f"Processing {test_name}...", end=" ")
    
    # Get questions for this test
    test_questions = questions_data.get(test_name, [])
    
    # Get answers from the three arrays
    signs_answers = signs_answers_all[test_index] if test_index < len(signs_answers_all) else []
    priorities_answers = priorities_answers_all[test_index] if test_index < len(priorities_answers_all) else []
    questions_answers = questions_answers_all[test_index] if test_index < len(questions_answers_all) else []
    
    # Build test object
    test_obj = {
        "test_id": test_name,
        "test_number": test_num,
        "sections": {
            "road_signs": [],
            "priorities": [],
            "general_questions": []
        }
    }
    
    
    # 1. Road Signs Section (16 signs)
    for sign_num in range(1, 17):
        # Find image path
        sign_image = f"{test_name}_sign_{sign_num:02d}.png"
        image_path = f"data/individual_signs/{test_name}/{sign_image}"
        
        # Find answer from parsed data (answers are plain strings in the array)
        answer = ""
        if sign_num <= len(signs_answers):
            answer = signs_answers[sign_num - 1]
        
        sign_obj = {
            "sign_number": sign_num,
            "image_path": image_path,
            "correct_answer": answer
        }
        test_obj["sections"]["road_signs"].append(sign_obj)
    
    # 2. Priorities Section (8 priorities)
    for priority_num in range(1, 9):
        # Find image path (relative to project root)
        priority_image = f"{test_name}_priority_{priority_num:02d}.png"
        image_path = f"data/individual_priorities/{test_name}/{priority_image}"
        
        # Find answer from parsed data (answers are plain strings in the array)
        answer = ""
        if priority_num <= len(priorities_answers):
            answer = priorities_answers[priority_num - 1]
        
        priority_obj = {
            "priority_number": priority_num,
            "image_path": image_path,
            "correct_answer": answer
        }
        test_obj["sections"]["priorities"].append(priority_obj)
    

    # 3. General Questions Section
    for i, q in enumerate(test_questions, 1):
        # Find answer from parsed data (answers are plain strings in the array)
        answer = ""
        if i <= len(questions_answers):
            answer = questions_answers[i - 1]
        
        question_obj = {
            "question_number": q.get("question_number", i),
            "question_ar": q.get("question_ar", ""),
            "question_fr": q.get("question_fr", ""),
            "correct_answer": answer
        }
        test_obj["sections"]["general_questions"].append(question_obj)

    final_data[test_name] = test_obj
    print(f"✓ 16 signs, 8 priorities, {len(test_questions)} questions,")

print(f"\n{'='*60}")
print(f"Processed {len(final_data)} tests")
print(f"{'='*60}")

Processing test-01... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-02... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-03... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-04... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-05... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-06... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-07... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-08... ✓ 16 signs, 8 priorities, 5 questions,
Processing test-09... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-10... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-11... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-12... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-13... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-14... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-15... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-16... ✓ 16 signs, 8 priorities, 6 questions,
Processing test-17... ✓ 

## Save Final JSON

In [4]:
# Save to file
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(final_data, f, ensure_ascii=False, indent=2)

print(f"✓ Saved to: {OUTPUT_FILE}")

# Calculate totals
total_questions = sum(len(test["sections"]["general_questions"]) for test in final_data.values())
total_signs = sum(len(test["sections"]["road_signs"]) for test in final_data.values())
total_priorities = sum(len(test["sections"]["priorities"]) for test in final_data.values())

print(f"\nTotals:")
print(f"  Tests: {len(final_data)}")
print(f"  Road Signs: {total_signs}")
print(f"  Priorities: {total_priorities}")
print(f"  General Questions: {total_questions}")
print(f"  Total Items: {total_questions + total_signs + total_priorities}")

✓ Saved to: ..\data\final_quiz_data.json

Totals:
  Tests: 20
  Road Signs: 320
  Priorities: 160
  General Questions: 119
  Total Items: 599


## View Sample Output

In [5]:
# Display sample from test-01
sample = final_data["test-01"]

print("="*60)
print("SAMPLE: test-01")
print("="*60)


print(f"\nRoad Signs ({len(sample['sections']['road_signs'])}):")
print(json.dumps(sample['sections']['road_signs'][:2], ensure_ascii=False, indent=2))

print(f"\nPriorities ({len(sample['sections']['priorities'])}):")
print(json.dumps(sample['sections']['priorities'][:2], ensure_ascii=False, indent=2))

print(f"\nGeneral Questions ({len(sample['sections']['general_questions'])}):")
print(json.dumps(sample['sections']['general_questions'][:2], ensure_ascii=False, indent=2))


SAMPLE: test-01

Road Signs (16):
[
  {
    "sign_number": 1,
    "image_path": "data/individual_signs/test-01/test-01_sign_01.png",
    "correct_answer": "1- حذار، خطر غير معين."
  },
  {
    "sign_number": 2,
    "image_path": "data/individual_signs/test-01/test-01_sign_02.png",
    "correct_answer": "2- الدوران إلى اليمين ممنوع."
  }
]

Priorities (8):
[
  {
    "priority_number": 1,
    "image_path": "data/individual_priorities/test-01/test-01_priority_01.png",
    "correct_answer": "1- محور دوراني مع إشارة ترك المرور - تمر السيارة الحمراء ثم الصفراء ."
  },
  {
    "priority_number": 2,
    "image_path": "data/individual_priorities/test-01/test-01_priority_02.png",
    "correct_answer": "2- تقاطع طرق مع إشارة قف على 150 متر و إشارة قف - تمر السيارتان الزرقاء و الصفراء في نفس الوقت ثم تمر السيارة الحمراء ."
  }
]

General Questions (6):
[
  {
    "question_number": 1,
    "question_ar": "بأي عامل ترتبط مسافة الأمان؟",
    "question_fr": "De quels facteurs dépend la distance de sécu

## Verify Data Completeness

In [6]:
# Check for missing data
print("Data Completeness Check:")
print("="*60)

for test_name, test_data in final_data.items():
    issues = []
    
    # Check questions
    questions = test_data["sections"]["general_questions"]
    empty_q = sum(1 for q in questions if not q["correct_answer"])
    if empty_q > 0:
        issues.append(f"{empty_q} questions missing answers")
    
    # Check signs
    signs = test_data["sections"]["road_signs"]
    empty_s = sum(1 for s in signs if not s["correct_answer"])
    if empty_s > 0:
        issues.append(f"{empty_s} signs missing answers")
    
    # Check priorities
    priorities = test_data["sections"]["priorities"]
    empty_p = sum(1 for p in priorities if not p["correct_answer"])
    if empty_p > 0:
        issues.append(f"{empty_p} priorities missing answers")
    
    if issues:
        print(f"⚠ {test_name}: {', '.join(issues)}")
    else:
        print(f"✓ {test_name}: Complete")

print("="*60)

Data Completeness Check:
✓ test-01: Complete
✓ test-02: Complete
✓ test-03: Complete
✓ test-04: Complete
✓ test-05: Complete
✓ test-06: Complete
✓ test-07: Complete
✓ test-08: Complete
✓ test-09: Complete
✓ test-10: Complete
✓ test-11: Complete
✓ test-12: Complete
✓ test-13: Complete
✓ test-14: Complete
✓ test-15: Complete
✓ test-16: Complete
✓ test-17: Complete
✓ test-18: Complete
✓ test-19: Complete
✓ test-20: Complete
